# Pipeline de Extracción de Datos de la EEA

Este *notebook* implementa un flujo completo de extracción, transformación y carga (ETL) para obtener datos históricos de la *European Environment Agency* (EEA). El objetivo es consolidar una base de datos sobre mediciones de contaminantes atmosféricos en las principales capitales europeas.

La API de la EEA permite acceder a series temporales sobre diferentes contaminantes atmosféricos de diversas ciudades. En cada ciudad de las que se disponen datos, se encuentran diferentes dispositivos que miden distintos contaminantes, por lo que para cada ciudad se dispone de datos sobre diferentes contaminantes.

En este proyecto, hemos optado por:

* Formato *parquet*, ya que es un formato de almacenamiento columnar que ofrece una compresión superior y un rendimiento de lectura mucho más rápido que el CSV tradicional. Además, es el formato en que la API devuelve los archivos.

* Granularidad: seleccionamos datos con agregación horaria, permitiendo análisis detallados de ciclos diarios.

* Alcance: tomamos los datos desde 2013 hasta 2024.

Por tanto, seguimos 5 fases para realizar este proceso ETL:

1. **Catalogación geográfica y mapeo de capitales**: no todos los países almacenan datos de la misma manera. Por ello, consultamos el catálogo de países de la API e implementamos un mapeo manual de capitales, teniendo como excepciones aquellos países sin capitales indexadas.

2. **Identificación de contaminantes (Pollutants)**: realizamos una petición a la API para obtener el catálogo de contaminantes monitorizados (como $NO_2$, $PM_{10}$, $O_3$, etc.). Esto asegura que el script sea compatible con cualquier nueva sustancia que la agencia comience a reportar en el futuro.

3. **Generación de consultas (*Payloads*)**: debido al gran volumen de información, la API no entrega los archivos directamente. Entonces, construimos solicitudes de red complejas (*JSON payloads*) que combinan: geolocalización (país + ciudad), parámetros técnicos (tipo de fuente y método de agregación) y un rango temporal. La API responde con una lista de URLs temporales para la descarga de los fragmentos de datos.

4. **Descarga optimizada y gestión de memoria**: para evitar el colapso de la memoria RAM al manejar cientos de archivos, implementamos una descarga por flujo de datos (*streaming*): los archivos se descargan en bloques de 8KB, se organiza el sistema de archivos en una jerarquía de carpetas por país ISO y se implementa un sistema de reintentos y verificación de existencia para evitar descargas duplicadas.

5. **Validación de esquemas y normalización final**: una vez en el disco, el script realiza una inspección profunda de cada archivo: verificamos qué contaminante contiene realmente cada archivo (detectando posibles inconsistencias en la API). Además, renombramos los archivos para que tengan una nomenclatura estándar: PAIS-CIUDAD-CONTAMINANTE_INDICE.parquet, facilitando su posterior uso. 

In [13]:
# ==============================================================================
# IMPORTACIÓN DE LIBRERÍAS Y DEPENDENCIAS
# ==============================================================================

from aiohttp import request           # Estándar para realizar peticiones HTTP (GET/POST) a la API de la EEA.
import requests
import os                             # Funciones básicas del sistema operativo.
from pathlib import Path              # Interfaz orientada a objetos para rutas (recomendado sobre os.path).
import pandas as pd                   # Motor principal para lectura de archivos Parquet y validación de columnas.
from tqdm.notebook import tqdm        # Visualización de barras de progreso.
from collections import defaultdict   # Diccionarios con inicialización automática.

In [14]:
# ==============================================================================
# CONFIGURACIÓN DE LA API - European Environment Agency (EEA)
# ==============================================================================

# Esta URL es la base para el servicio de descarga de datos de la EEA.
# Se utiliza como prefijo para todas las peticiones a la API.
BASE_URL = "https://eeadmz1-downloads-api-appservice.azurewebsites.net"

# ENDPOINTS DISPONIBLES:
# ----------------------
# /Country:           Obtiene el listado de países disponibles.
# /City:              Obtiene el listado de ciudades filtradas por país.
# /Pollutant:         Lista de contaminantes atmosféricos monitoreados.
# /ParquetFile/urls:  Punto de acceso principal para obtener las rutas de descarga de los archivos de datos en formato Parquet.

In [19]:
# ==============================================================================
# OBTENCIÓN DE CATÁLOGO DE PAÍSES
# ==============================================================================

# Realizamos una petición GET al endpoint "/Country" para obtener la lista oficial de naciones registradas en la base de datos de la EEA.
r = requests.get(f"{BASE_URL}/Country")

# Convertimos la respuesta JSON en una lista de diccionarios.
# Cada diccionario contiene metadatos del país (nombre, código, etc.).
countries = r.json()

# PROCESAMIENTO DE DATOS:
# Extraemos únicamente el identificador "countryCode" de cada objeto para facilitar iteraciones y filtrados posteriores.
country_codes = [c["countryCode"] for c in countries]

In [6]:
# ==============================================================================
# MAPEO DE METADATOS: PAÍSES Y CIUDADES CAPITALES
# ==============================================================================

# Diccionario de referencia que vincula el código ISO del país con su capital. Hay países sin capital definida (None).
capitals_europe = {
    "AD": None,                 # Andorra
    "AL": None,                 # Albania
    "AT": "Wien",               # Austria
    "BA": None,                 # Bosnia y Herzegovina
    "BE": "Bruxelles_Brussel",  # Bélgica
    "BG": "Sofia",              # Bulgaria
    "CH": "Bern",               # Suiza
    "CY": "Lefkosia",           # Chipre
    "CZ": "Praha",              # Chequia
    "DE": "Berlin",             # Alemania
    "DK": "Kobenhavn",          # Dinamarca
    "EE": "Tallinn",            # Estonia
    "ES": "Madrid",             # España
    "FI": "Helsinki",           # Finlandia
    "FR": "Paris",              # Francia
    "GR": "Athina",             # Grecia
    "HR": "Zagreb",             # Croacia
    "HU": "Budapest",           # Hungría
    "IE": "Dublin",             # Irlanda
    "IS": "Reykjavik",          # Islandia
    "IT": "Roma",               # Italia
    "LT": "Vilnius",            # Lituania
    "LU": "Luxembourg",         # Luxemburgo
    "LV": "Riga",               # Letonia
    "ME": None,                 # Montenegro
    "MK": None,                 # Macedonia del Norte
    "MT": "Valletta",           # Malta
    "NL": "Amsterdam",          # Países Bajos
    "NO": "Oslo",               # Noruega
    "PL": "Warszawa",           # Polonia
    "PT": "Lisboa",             # Portugal
    "RO": "Bucuresti",          # Rumanía
    "RS": None,                 # Serbia
    "SE": "Stockholm",          # Suecia
    "SI": "Ljubljana",          # Eslovenia
    "SK": "Bratislava",         # Eslovaquia
    "XK": None                  # Kosovo
}

In [ ]:
# ==============================================================================
# FILTRADO DE CIUDADES: OBTENCIÓN EXCLUSIVA DE CAPITALES
# ==============================================================================

# "all_cities" almacenará el mapeo final: { "CÓDIGO_PAÍS": ["Nombre_Capital"] }
all_cities = {}

print("Fetching cities and filtering only capitals...")

# Iteramos por cada código de país.
# Utilizamos "tqdm" para visualizar una barra de progreso, útil dado que se realiza una petición de red por cada país.
for cc in tqdm(country_codes):
    
    # La API requiere un método POST para consultar ciudades.
    # El cuerpo de la petición (payload) debe ser una lista con el código del país: [cc]
    r = requests.post(f"{BASE_URL}/City", json=[cc])
    city_list = r.json()

    # Recuperamos el nombre de la capital definida en nuestro diccionario "capitals_europe".
    capital_name = capitals_europe.get(cc)
    
    if capital_name:
        # Si tenemos definida una capital para este país, buscamos su coincidencia exacta dentro de la lista de ciudades que nos ha devuelto la API.
        all_cities[cc] = [
            ci["cityName"] for ci in city_list 
            if ci["cityName"] == capital_name
        ]
    else:
        # Si el país no tiene capital definida en nuestro mapeo (None), inicializamos la entrada con una lista vacía para evitar errores de clave.
        all_cities[cc] = []

# Calculamos el total de ciudades capturadas sumando la longitud de todas las listas de valores.
total_cities = sum(len(v) for v in all_cities.values())
print(f"Total capitals collected")

Total capitals collected


In [ ]:
# ==============================================================================
# OBTENCIÓN DE CATÁLOGO DE CONTAMINANTES (POLLUTANTS)
# ==============================================================================

print("Fetching pollutants list...")

# Solicitamos a la API el listado completo de sustancias monitorizadas por la EEA.
# El endpoint "/Pollutant" devuelve objetos con el nombre completo y su abreviatura técnica.
r = requests.get(f"{BASE_URL}/Pollutant")

# Parseamos la respuesta a una estructura de lista/diccionario.
pollutants = r.json()

# Extraemos el campo "notation", que es el identificador técnico.
# Este código es estrictamente necesario para parametrizar las futuras URLs de descarga.
pollutant_codes = [p["notation"] for p in pollutants]

# Confirmamos cuántos elementos se han cargado correctamente.
print(f"Found {len(pollutant_codes)} pollutants.")

Fetching pollutants list...
Found 648 pollutants.


In [ ]:
# ==============================================================================
# GESTIÓN DEL SISTEMA DE ARCHIVOS (DIRECTORIO DE DATOS)
# ==============================================================================

# Definimos la ruta del directorio donde se descargarán y almacenarán los archivos.
# Usamos "Path" de la librería "pathlib".
DATA_DIR = Path("eea_parquet")

# Creamos la carpeta físicamente en el disco.
# Parámetros:
# - exist_ok=True: si la carpeta ya existe, no lanza un error.
# - parents=True (opcional): crearía también rutas intermedias si las hubiera.
DATA_DIR.mkdir(exist_ok=True)

# Nota: A partir de este punto, todas las descargas se enviarán a: ./{DATA_DIR}/

In [8]:
# ==============================================================================
# GENERACIÓN DE SOLICITUDES Y OBTENCIÓN DE ENLACES DE DESCARGA (PARQUET)
# ==============================================================================

# Lista donde guardaremos objetos con la URL y sus metadatos asociados.
all_files = []

# La API requiere un correo electrónico para registrar la solicitud de datos.
email = "email@gmail.com" 

# Estructura: Por cada País -> Por cada Capital -> Por cada Contaminante.
for country_code, cities in tqdm(all_cities.items(), desc="Procesando países"):
    
    # Si el país no tiene ciudades (lista vacía), saltamos a la siguiente iteración.
    if not cities:
        continue
        
    for city in cities:
        for pollutant in pollutant_codes:
            
            # CONFIGURACIÓN DEL PAYLOAD:
            # Definimos los parámetros de filtrado para la API de archivos Parquet.
            payload = {
                "countries": [country_code],
                "cities": [city],
                "pollutants": [pollutant],
                "dataset": 2,                            # Identificador del dataset (específico de EEA), es el de datos validados.
                "source": "EEA",                         # Fuente de los datos
                "method": "Automatic",                   # Método de adquisición
                "dateTimeStart": "2013-01-01T00:00:00Z", # Rango temporal inicio
                "dateTimeEnd": "2024-12-31T23:59:59Z",   # Rango temporal fin
                "aggregationType": "hour",               # Granularidad de los datos (horaria)
                "email": email,
                "compress": False                        # No comprimir para obtener el .parquet directo
            }

            # Realizamos la petición POST para generar las URLs de descarga.
            r = requests.post(f"{BASE_URL}/ParquetFile/urls", json=payload)
            
            # TRATAMIENTO DE LA RESPUESTA:
            # 1. lstrip("\ufeff"): eliminamos el "Byte Order Mark" (BOM) que la API puede incluir al inicio de la respuesta y que rompería el parseo.
            text = r.text.lstrip("\ufeff") 
            
            # 2. splitlines(): la API devuelve una cadena de texto con una URL por línea. Limpiamos espacios y filtramos líneas vacías.
            urls = [u.strip() for u in text.splitlines() if u.strip()]

            # ALMACENAMIENTO TÉCNICO:
            # Asociamos cada URL encontrada con su contexto original para facilitar la organización de archivos durante la descarga física.
            for url in urls:
                all_files.append({
                    "url": url,
                    "country": country_code,
                    "city": city,
                    "pollutant": pollutant
                })

  0%|          | 0/38 [00:00<?, ?it/s]

In [9]:
# ==============================================================================
# EJECUCIÓN DE DESCARGAS Y ORGANIZACIÓN EN DISCO
# ==============================================================================

# "counter" rastrea cuántos archivos existen para una combinación país-ciudad-contaminante.
# Evita colisiones de nombres si una consulta devuelve múltiples fragmentos.
counter = defaultdict(int)

for f in tqdm(all_files, desc="Descargando archivos Parquet"):
    # Aseguramos que la URL esté libre de caracteres invisibles (BOM) y sea válida.
    url = f["url"].lstrip("\ufeff").strip()
    if not url.startswith("http"):
        continue

    country = f["country"]
    # Reemplazamos espacios por guiones bajos para evitar problemas en sistemas de archivos.
    city = f["city"].replace(" ", "_")
    pollutant = f["pollutant"].replace(" ", "_")

    # Creamos una subcarpeta por país para mantener el orden (ej. eea_parquet/ES/).
    country_dir = DATA_DIR / country
    country_dir.mkdir(exist_ok=True)

    # Usamos un índice incremental por cada combinación única de parámetros.
    key = f"{country}_{city}_{pollutant}"
    counter[key] += 1
    index = counter[key]

    # Formato final: PAIS-CIUDAD-CONTAMINANTE-INDICE.parquet
    filename = f"{country}-{city}-{pollutant}-{index}.parquet"
    path = country_dir / filename

    # Saltamos la descarga si el archivo ya existe localmente.
    if path.exists():
        continue

    try:
        # stream=True: no carga el archivo completo en la RAM, sino que lo procesa por partes.
        # timeout=60: evita que el script se quede colgado indefinidamente si falla la red.
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status() # Lanza error si la respuesta no es 200 (OK).

            # Abrimos el archivo local en modo "escritura binaria" (wb).
            with open(path, "wb") as out_file:
                # Escribimos el contenido en bloques (chunks) de 8KB.
                # Ideal para manejar archivos grandes sin saturar la memoria del sistema.
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk: # Filtrar posibles keep-alive chunks
                        out_file.write(chunk)
                        
    except Exception as e:
        # Captura errores de red, permisos o timeouts para que el script no se detenga.
        print(f"Error descargando {url}: {e}")

Descargando archivos Parquet:   0%|          | 0/15242 [00:00<?, ?it/s]

In [10]:
# ==============================================================================
# POST-PROCESAMIENTO: NORMALIZACIÓN Y RENOMBRADO BASADO EN CONTENIDO
# ==============================================================================

# Lista de posibles nombres de columna que la EEA utiliza para identificar el contaminante.
POLLUTANT_COLUMNS = ["pollutant", "Pollutant", "pollutantNotation", "notation"]

# Diccionario para controlar la numeración de archivos y evitar sobrescrituras.
name_counter = defaultdict(int)

# Iteramos sobre cada carpeta de país creada en el paso anterior.
for country_folder in tqdm([p for p in DATA_DIR.iterdir() if p.is_dir()], desc="Normalizando nombres"):
    country = country_folder.name
    
    # Procesamos todos los archivos .parquet dentro de la carpeta del país.
    for parquet_file in country_folder.glob("*.parquet"):
        try:
            # Leemos solo los metadatos/cabecera para identificar las columnas.
            df = pd.read_parquet(parquet_file)
        except Exception:
            # Si el archivo está corrupto o no es un Parquet válido, lo saltamos.
            continue

        # Buscamos cuál de los nombres en POLLUTANT_COLUMNS existe en el DataFrame.
        pollutant_col = next((col for col in POLLUTANT_COLUMNS if col in df.columns), None)
        
        if pollutant_col is None:
            continue # Si no se encuentra información del contaminante, pasamos al siguiente.

        # Obtenemos los valores únicos de contaminantes presentes en el archivo.
        pollutants = df[pollutant_col].dropna().unique()
        n_pollutants = len(pollutants)
        
        if n_pollutants == 0:
            continue
            
        base_pollutant = str(pollutants[0])

        # LÓGICA DE NOMBRADO:
        # 1. Si el archivo tiene un solo contaminante: "ES-NO2"
        # 2. Si es un archivo mixto: "ES-NO2(3)" (indicando que hay 3 tipos de datos dentro)
        base_name = f"{country}-{base_pollutant}" if n_pollutants == 1 else f"{country}-{base_pollutant}({n_pollutants})"
        
        # GESTIÓN DE DUPLICADOS:
        # Si ya existe un "ES-NO2", el siguiente será "ES-NO2_2", y así sucesivamente.
        name_counter[base_name] += 1
        idx = name_counter[base_name]

        # Construcción del nombre final y ruta.
        final_name = f"{base_name}.parquet" if idx == 1 else f"{base_name}_{idx}.parquet"
        new_path = parquet_file.parent / final_name

        # Ejecutamos el renombrado físico si el destino no existe ya.
        if not new_path.exists():
            parquet_file.rename(new_path)

Normalizando nombres:   0%|          | 0/22 [00:00<?, ?it/s]

Tras tener los archivos descargados en formato *parquet*, los procesamos para poder utilizarlos en procesos futuros. Por ello, llevabamos a cabo las siguientes acciones:

* **Conversión de formato**: transformamos los archivos de *parquet* a CSV.

* **Filtrado de variables**: seleccionamos únicamente las variables con información relevante: *SamplingPoint*, *Start*, *End*, *Pollutant*, *Value* y *Unit*, eliminando metadatos técnicos redundantes de la API y reduciendo el consumo de memoria.

* **Mantenimiento de Jerarquía**: el script replica automáticamente la estructura de carpetas por país, asegurando que los datos sigan organizados geográficamente tras la conversión.

In [ ]:
# ==============================================================================
# CONVERSIÓN: PARQUET A CSV 
# ==============================================================================

# Definimos las rutas absolutas para el origen de datos y el destino procesado.
# El uso de cadenas "raw" (r"") evita problemas con las barras invertidas de Windows.
ruta_origen = r'C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\eea_parquet'
ruta_destino = r'C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_csv'

# FEATURE SELECTION
# ------------------------------------------------------------------------------
# Definimos las variables con las que nos queremos quedar.
# Esto reduce drásticamente el peso de los archivos CSV resultantes.
columnas_tfm = ['SamplingPoint', 'Start', 'End', 'Pollutant', 'Value', 'Unit']

# os.walk permite navegar por toda la jerarquía de carpetas por país.
for raiz, carpetas, archivos in os.walk(ruta_origen):
    for nombre_archivo in archivos:
        # Procesamos únicamente archivos con extensión .parquet
        if nombre_archivo.endswith('.parquet'):
            
            ruta_completa_entrada = os.path.join(raiz, nombre_archivo)
            
            # Replicamos la estructura de subcarpetas (por país) en el destino
            ruta_relativa = os.path.relpath(raiz, ruta_origen)
            carpeta_final_destino = os.path.join(ruta_destino, ruta_relativa)
            
            # Creamos la carpeta de destino si no existe (ej. la carpeta "ES", "FR", etc.)
            if not os.path.exists(carpeta_final_destino):
                os.makedirs(carpeta_final_destino)
            
            # Definimos el nuevo nombre y ruta de salida
            nombre_csv = nombre_archivo.replace('.parquet', '.csv')
            ruta_completa_salida = os.path.join(carpeta_final_destino, nombre_csv)
            
            try:
                # Cargamos el archivo parquet.
                df = pd.read_parquet(ruta_completa_entrada)

                # Solo intentamos seleccionar columnas que realmente existan en el archivo, evitando errores si la API cambió el nombre de alguna variable.
                columnas_presentes = [col for col in columnas_tfm if col in df.columns]
                df_filtrado = df[columnas_presentes]
                
                # Exportamos a CSV:
                # index=False evita crear una columna adicional de índices innecesaria.
                df_filtrado.to_csv(ruta_completa_salida, index=False)
                
            except Exception as e:
                # Registro de errores por archivo (evita que el script se detenga)
                print(f"Error procesando {nombre_archivo}: {e}")

print("\n¡Conversión y filtrado completados!")


¡Conversión y filtrado completados!


Para finalizar el pre-procesamiento de estos archivos, unificamos los datos a nivel de contaminante. Es decir, para una misma ciudad, hay varios archivos sobre el mismo contaminante (*pollutant*), y eso se debe a que se registra el mismo contaminante a la misma hora en diferentes puntos de una misma ciudad. Por lo tanto, unificamos todos esos archivos para un mismo contaminante en una misma ciudad, siguiendo los siguientes pasos:

* **Tratamiento de multi-estaciones**: en casos donde existen múltiples puntos de muestreo sobre un mismo contaminante para una misma ciudad, aplicamos una agregación mediante la media aritmética por cada marca de tiempo (*Start*). Esto genera una serie temporal única y representativa por contaminante.

* **Optimización del *workspace***: limpiamos el almacenamiento eliminando los archivos fragmentados originales y conservando únicamente los archivos consolidados (*Unificado_POLLUTANT.csv*), optimizando así el espacio en disco y la claridad para la fase de análisis.

In [ ]:
# ==============================================================================
# UNIFICACIÓN DE SERIES TEMPORALES CON MISMO CONTAMINANTE
# ==============================================================================

ruta_csv = r'C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_csv'

for raiz, carpetas, archivos in os.walk(ruta_csv):
    # Filtramos solo archivos CSV para evitar procesar otros formatos
    archivos_csv = [f for f in archivos if f.endswith('.csv')]
    
    if not archivos_csv:
        continue

    datos_subcarpeta = []

    for f in archivos_csv:
        ruta_archivo = os.path.join(raiz, f)
        df_temp = pd.read_csv(ruta_archivo)
        datos_subcarpeta.append(df_temp)
    
    if datos_subcarpeta:
        # Concatenamos todos los archivos del país con el mismo contaminante en un único DataFrame
        df_total = pd.concat(datos_subcarpeta, ignore_index=True)
        
        # Identificamos los contaminantes únicos presentes
        pollutants = df_total['Pollutant'].unique()
        
        for poll in pollutants:
            # Filtramos el DataFrame para aislar el contaminante actual
            df_poll = df_total[df_total['Pollutant'] == poll]
            
            # Si existen múltiples estaciones para la misma hora ('Start'), calculamos el valor promedio. Esto unifica la tendencia de la ciudad.
            df_agrupado = df_poll.groupby('Start').agg({
                'Value': 'mean',      # Calculamos la media de las mediciones
                'End': 'first',        # Mantenemos el resto de metadatos constantes
                'Pollutant': 'first',
                'Unit': 'first',
            }).reset_index()
            
            nombre_final = f"Unificado_{poll}.csv"
            df_agrupado.to_csv(os.path.join(raiz, nombre_final), index=False)

    # Una vez creado el archivo unificado, eliminamos los archivos fuente individuales para mantener solo el dataset listo para el análisis.
    for f in archivos_csv:
        if not f.startswith("Unificado_"):
            os.remove(os.path.join(raiz, f))

print("\n¡Proceso de unificación y limpieza completado!")


¡Proceso de unificación y limpieza completado!


Así, finalmente tenemos para cada país/ciudad un CVS por cada contaminante del que se tiene datos, y cada CSV contiene las variables:

1. *Start*: fecha de comienzo de medición.
2. *End*: fecha de fin de medición (1 hora después).
3. *Pollutant*: nombre del contaminante. Están clasificados numéricamente.
4. *Value*: el valor del contaminante.
5. *Unit*: unidad en la que está medido el contaminante.